In [ ]:
from google.colab import drive
import os

# montar drive
drive.mount('/content/drive')

# carpeta del proyecto en drive
WORK_DIR = '/content/drive/MyDrive/text2sql_project'

# crear estructura de directorios
for subdir in ['data', 'checkpoints', 'models', 'results']:
    os.makedirs(f'{WORK_DIR}/{subdir}', exist_ok=True)

os.chdir(WORK_DIR)

In [ ]:
!pip install -q transformers==4.46.0 datasets==2.19.0 accelerate sqlparse sentencepiece

In [ ]:
import json
import copy
from tqdm import tqdm

SPIDER_DATA_DIR = f"{WORK_DIR}/data/spider_data"
DB_BASE_PATH = f"{SPIDER_DATA_DIR}/database"

# cargar archivos json
with open(f"{SPIDER_DATA_DIR}/train_spider.json", 'r', encoding='utf-8') as f: train_spider = json.load(f)
with open(f"{SPIDER_DATA_DIR}/train_others.json", 'r', encoding='utf-8') as f: train_others = json.load(f)
with open(f"{SPIDER_DATA_DIR}/dev.json", 'r', encoding='utf-8') as f: dev_data = json.load(f)
with open(f"{SPIDER_DATA_DIR}/tables.json", 'r', encoding='utf-8') as f: tables_data = json.load(f)

train_data = train_spider + train_others

tables_dict = {t['db_id']: t for t in tables_data}

def enriquecer_ejemplo(ejemplo, tables_dict):
    """Agrega la info del esquema (tablas, columnas, PKs, FKs) al ejemplo"""
    db_id = ejemplo['db_id']
    if db_id not in tables_dict: return ejemplo

    table_info = tables_dict[db_id]
    ej = copy.deepcopy(ejemplo)
    ej['db_table_names'] = table_info['table_names_original']
    ej['db_column_names'] = table_info['column_names_original']
    ej['db_column_types'] = table_info['column_types']
    ej['db_primary_keys'] = table_info['primary_keys']
    ej['db_foreign_keys'] = table_info['foreign_keys']
    return ej

dataset_bruto = {
    'train': [enriquecer_ejemplo(ej, tables_dict) for ej in train_data],
    'validation': [enriquecer_ejemplo(ej, tables_dict) for ej in dev_data]
}
print("✅ Dataset Spider cargado y enriquecido.")

In [ ]:
def serializar_esquema(ejemplo):
    db_table_names = ejemplo["db_table_names"]
    db_column_names = ejemplo["db_column_names"]
    db_column_types = ejemplo["db_column_types"]
    db_primary_keys = ejemplo["db_primary_keys"]
    db_foreign_keys = ejemplo["db_foreign_keys"]

    tablas_dict = {i: [] for i in range(len(db_table_names))}

    # agrupar columnas por tabla con sus tipos
    for col_idx, (table_idx, col_name) in enumerate(db_column_names):
        if table_idx == -1: continue # ignorar asterisco

        col_str = f"{db_column_types[col_idx]} {col_name}"
        if col_idx in db_primary_keys: col_str += " (pk)"
        tablas_dict[table_idx].append(col_str)

    # construir el string del esquema
    table_parts = []
    for table_idx in range(len(db_table_names)):
        table_name = db_table_names[table_idx]
        columnas_str = " , ".join(tablas_dict[table_idx])
        table_parts.append(f"{table_name} : {columnas_str}")

    schema_str = " | ".join(table_parts)

    # agregar foreign keys
    if db_foreign_keys:
        fk_parts = []
        for fk_from, fk_to in db_foreign_keys:
            t_from = db_column_names[fk_from][0]
            t_to = db_column_names[fk_to][0]
            col_from = db_column_names[fk_from][1]
            col_to = db_column_names[fk_to][1]
            fk_parts.append(f"{db_table_names[t_from]}.{col_from} = {db_table_names[t_to]}.{col_to}")

        if fk_parts:
            schema_str += f" | foreign keys: {', '.join(fk_parts)}"

    return schema_str

def preparar_ejemplo(ejemplo):
    schema_str = serializar_esquema(ejemplo)
    input_text = f"translate to SQL: {ejemplo['question'].strip()} | db_id: {ejemplo['db_id']} | schema: {schema_str}"

    return {
        "input": input_text,
        "output": ejemplo["query"].strip(),
        "db_id": ejemplo["db_id"]
    }

train_prepared = [preparar_ejemplo(ej) for ej in tqdm(dataset_bruto["train"], desc="Train")]
val_prepared = [preparar_ejemplo(ej) for ej in tqdm(dataset_bruto["validation"], desc="Val")]

In [ ]:
from datasets import Dataset as HFDataset
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("Salesforce/codet5p-770m")

train_dataset = HFDataset.from_list(train_prepared)
val_dataset = HFDataset.from_list(val_prepared)

def tokenizar_ejemplos(examples):
    model_inputs = tokenizer(examples["input"], max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(examples["output"], max_length=128, truncation=True, padding="max_length")

    # reemplazar padding por -100 para ignorar en la loss
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

train_tokenized = train_dataset.map(tokenizar_ejemplos, batched=True, num_proc=4, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenizar_ejemplos, batched=True, remove_columns=val_dataset.column_names)

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5p-770m").to(device)

training_args = TrainingArguments(
    output_dir=f"{WORK_DIR}/checkpoints_spider",
    num_train_epochs=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=100,
    dataloader_num_workers=2,
    remove_unused_columns=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

# guardar el modelo final
final_model_path = f"{WORK_DIR}/models/codet5p_spider_final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

In [ ]:
import os
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# cambiar las XXXX por el numero del checkpoint
CHECKPOINT_NUM = "XXXX"
checkpoint_path = f"{WORK_DIR}/checkpoints_spider/checkpoint-{CHECKPOINT_NUM}"

if os.path.exists(checkpoint_path):
    # mismos parametros exactos que cuando arranco el entrenamiento
    training_args = TrainingArguments(
        output_dir=f"{WORK_DIR}/checkpoints_spider",
        num_train_epochs=8,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        optim="adamw_torch",
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        logging_steps=100,
        dataloader_num_workers=2,
        remove_unused_columns=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # reanudar entrenamiento
    trainer.train(resume_from_checkpoint=checkpoint_path)

    # guardar el modelo final
    final_model_path = f"{WORK_DIR}/models/codet5p_spider_final"
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)
else:
    print(f"Error: No se encontró la carpeta: {checkpoint_path}")

In [ ]:
import sqlite3
import sqlparse
import signal
import torch
import os
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, RobertaTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

# cargar modelo a evaluar
model_path = f"{WORK_DIR}/checkpoints_spider/checkpoint-800"
model_eval = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
model_eval.eval()

# cargar tokenizador
tokenizer_eval = RobertaTokenizer.from_pretrained("Salesforce/codet5p-770m")

class TimeoutError(Exception): pass
def timeout_handler(signum, frame): raise TimeoutError("Query timeout")

def generar_sql_agresivo(input_text, db_path):
    inputs = tokenizer_eval(input_text, max_length=512, truncation=True, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_length=256,
            num_beams=5,
            num_return_sequences=5,
            early_stopping=True
        )

    opciones_sql = [tokenizer_eval.decode(out, skip_special_tokens=True) for out in outputs]

    for sql in opciones_sql:
        try:
            parsed = sqlparse.parse(sql)
            if not (len(parsed) > 0 and str(parsed[0]).strip() != ''): continue

            signal.signal(signal.SIGALRM, timeout_handler)
            signal.alarm(2)
            conn = sqlite3.connect(db_path, timeout=1)
            conn.cursor().execute(sql)
            conn.close()
            signal.alarm(0)
            return sql # retorna la primera de las que funcione
        except:
            signal.alarm(0)
            continue

    return opciones_sql[0]

def comparar_resultados(sql_pred, sql_real, db_path):
    try:
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(3)
        conn = sqlite3.connect(db_path)
        c1, c2 = conn.cursor(), conn.cursor()
        c1.execute(sql_pred); r1 = set(tuple(row) for row in c1.fetchall())
        c2.execute(sql_real); r2 = set(tuple(row) for row in c2.fetchall())
        conn.close()
        signal.alarm(0)
        return r1 == r2
    except:
        signal.alarm(0)
        return False

correctos, total = 0, 0
predicciones_oficiales = []

for i, ejemplo in enumerate(tqdm(val_prepared, desc="Evaluando")):
    db_path = f"{DB_BASE_PATH}/{ejemplo['db_id']}/{ejemplo['db_id']}.sqlite"

    if not os.path.exists(db_path):
        predicciones_oficiales.append("SELECT 1")
        total += 1; continue

    sql_pred = generar_sql_agresivo(ejemplo['input'], db_path)
    predicciones_oficiales.append(sql_pred.replace("\n", " ").strip())

    if comparar_resultados(sql_pred, ejemplo['output'], db_path):
        correctos += 1
    total += 1

print(f"\n execution accuracy: {correctos}/{total} ({(correctos/total)*100:.2f}%)")

txt_path = f"{WORK_DIR}/results/predicciones_spider.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("\n".join(predicciones_oficiales))

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

%cd /content/
!rm -rf spider
!git clone https://github.com/taoyds/spider

!python /content/spider/evaluation.py \
    --gold /content/drive/MyDrive/text2sql_project/data/spider_data/dev_gold.sql \
    --pred /content/drive/MyDrive/text2sql_project/results/predicciones_spider.txt \
    --db /content/drive/MyDrive/text2sql_project/data/spider_data/database \
    --table /content/drive/MyDrive/text2sql_project/data/spider_data/tables.json \
    --etype all

In [ ]:
# Descomentar para subir el modelo a Hugging Face

# from huggingface_hub import HfApi
# from google.colab import userdata

# # Configurar token de HF
# HF_TOKEN = userdata.get("HF_TOKEN")
# REPO_NAME = "brunnoconti/codet5-sql-generator"

# # Subir modelo
# api = HfApi(token=HF_TOKEN)
# api.upload_folder(
#     folder_path="/content/drive/MyDrive/text2sql_project/checkpoints_spider/checkpoint-800",
#     repo_id=REPO_NAME,
#     repo_type="model",
# )